# Divar Real Estate - Project 1

Analysis of ~1,000,000 Divar real estate listings, following the structure and questions in `QBC12 _ AI _ Project 1.md`.

1. Data exploration and schema check
2. Preprocessing pipeline (Persian text, amenities, financial features, imputation)
3. Descriptive statistics (Q1-Q9)
4. Hypothesis tests (H1-H4)

Each function is tagged with `@author` for whoever built that piece of logic.

In [ ]:
import re
import os
import json as js
import warnings

import numpy as np
import pandas as pd
import jdatetime
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.path import Path
import seaborn as sns

import folium
from folium.plugins import HeatMap

from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.1f}".format

## 1. Data exploration and schema check

In [ ]:
df = pd.read_csv("Divar.csv", low_memory=False)
print("shape:", df.shape)
df.info()

Schema check: verifying the 61 columns are named and ordered correctly (scraped data occasionally has shifted headers). They line up fine here. One real naming issue: `has_restroom` holds text (`squat`/`seat`/`unselect`), not True/False, so it is renamed to `restroom_type`. Latitude/longitude get validated against Iran's bounding box and renamed for shorter code.

In [ ]:
EXPECTED_NAMED = {"has_security_guard", "has_barbecue", "building_direction",
                  "has_pool", "has_jacuzzi", "has_sauna", "floor_material"}
TAIL_SCHEMA = ["restroom_type", "has_security_guard", "has_barbecue", "building_direction",
               "has_pool", "has_jacuzzi", "has_sauna", "floor_material"]

def realign_schema(df):
    """Verify column names/order; fall back to a positional fix only if the header looks corrupt."""
    out = df.copy()
    if "Unnamed: 0" in out.columns:
        out = out.drop(columns=["Unnamed: 0"])
    present = EXPECTED_NAMED.intersection(out.columns)
    if len(present) == len(EXPECTED_NAMED):
        if "has_restroom" in out.columns and "restroom_type" not in out.columns:
            out = out.rename(columns={"has_restroom": "restroom_type"})
    else:
        new_cols = list(out.columns)
        new_cols[-len(TAIL_SCHEMA):] = TAIL_SCHEMA
        out.columns = new_cols
    rename_geo = {}
    if "latitude" not in out.columns:
        cand = [c for c in out.columns if "lat" in c.lower()]
        if cand: rename_geo[cand[0]] = "latitude"
    if "longitude" not in out.columns:
        cand = [c for c in out.columns if "lon" in c.lower()]
        if cand: rename_geo[cand[0]] = "longitude"
    out = out.rename(columns=rename_geo)
    if {"latitude", "longitude"}.issubset(out.columns):
        lat = pd.to_numeric(out["latitude"], errors="coerce")
        lon = pd.to_numeric(out["longitude"], errors="coerce")
        valid = lat.between(24, 40) & lon.between(44, 64)
        out["latitude"] = lat.where(valid)
        out["longitude"] = lon.where(valid)
    return out

df = realign_schema(df)
print("shape after schema check:", df.shape)

## 2. Preprocessing pipeline

Each step is a standalone function so it can be reused or skipped later, e.g. in a machine-learning pipeline. Rule: raw columns are only overwritten if broken in their original form (Persian numerals, text room counts); everything else is added as a new column.

### 2.1 Column pruning (@author: Erfan)

In [ ]:
def drop_high_null_columns(df, threshold=97.1943):
    """Drop columns that are almost entirely empty."""
    df = df.copy()
    null_pct = df.isnull().mean() * 100
    to_drop = null_pct[null_pct > threshold].index.tolist()
    print(f"dropping {len(to_drop)} high-null columns: {to_drop}")
    return df.drop(columns=to_drop)

def drop_unnecessary_columns(df):
    df = df.copy()
    to_drop = [c for c in ["rent_mode", "credit_mode", "floor_material"] if c in df.columns]
    print(f"dropping {to_drop}")
    return df.drop(columns=to_drop)

df = drop_high_null_columns(df)
df = drop_unnecessary_columns(df)
print("shape now:", df.shape)

### 2.2 Persian digits, room counts, construction year (@author: Mesbah, with Erfan's NLP fallback)

In [ ]:
_PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
_ARABIC_DIGITS = "٠١٢٣٤٥٦٧٨٩"
_DIGIT_MAP = {ord(p): str(i) for i, p in enumerate(_PERSIAN_DIGITS)}
_DIGIT_MAP.update({ord(a): str(i) for i, a in enumerate(_ARABIC_DIGITS)})

def fa_to_en_digits(value):
    """Persian/Arabic digits -> ASCII digits."""
    if pd.isna(value):
        return value
    return str(value).translate(_DIGIT_MAP)

ROOMS_MAP = {"بدون اتاق": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج یا بیشتر": 5}

def map_rooms_count(df, col="rooms_count"):
    out = df.copy()
    out["rooms_count"] = out[col].astype("string").str.strip().map(ROOMS_MAP).astype("Int64")
    return out

df = map_rooms_count(df)
print(df["rooms_count"].value_counts(dropna=False).sort_index())

In [ ]:
def _extract_year_from_description(text):
    """Guess a construction year from listing text when the structured field is empty."""
    if pd.isna(text):
        return np.nan
    text = fa_to_en_digits(str(text))
    if any(kw in text for kw in ["نوساز", "کلید نخورده", "کلیدنخورده", "صفر"]):
        return 1403.0
    m = re.search(r"ساخت\s*(1[34]\d{2})", text) or re.search(r"(1[34]\d{2})\s*ساخت", text)
    return float(m.group(1)) if m else np.nan

def clean_construction_year(df, col="construction_year"):
    out = df.copy()
    raw = out[col].astype("string").map(fa_to_en_digits)
    out["is_pre_threshold_year"] = raw.str.contains("قبل", na=False).fillna(False)
    year = raw.str.extract(r"(\d{4})", expand=False)
    out["construction_year"] = pd.to_numeric(year, errors="coerce")
    missing = out["construction_year"].isna()
    if missing.any() and "description" in out.columns:
        out.loc[missing, "construction_year"] = out.loc[missing, "description"].map(_extract_year_from_description)
    out["construction_year"] = out["construction_year"].clip(lower=1364.0, upper=1403.0)
    return out

df = clean_construction_year(df)
print(df["construction_year"].describe())

### 2.3 Amenities: keyword search with negation check (@author: Erfan & Mesbah)

A keyword like "استخر" (pool) only counts as True if it is not right after a negation word such as "بدون" (without) or "فاقد" (lacking) - "بدون استخر" (no pool) must not flip the amenity to True.

In [ ]:
AMENITY_REGEX = {
    "has_pool":           r"استخر|مجموعه آبی|استخردار",
    "has_sauna":          r"سونا",
    "has_jacuzzi":        r"جکوزی",
    "has_security_guard": r"نگهبان|حراست|سرایدار|لابی‌?من|دوربین مداربسته",
    "has_barbecue":       r"باربیکیو|کباب.?پز|آتشکده",
    "has_elevator":       r"آسانسور|اسانسور|لاین آسانسور",
    "has_balcony":        r"بالکن|تراس|ایوان|روف گاردن|بهار.?خواب",
    "has_gas":            r"گاز|فول امکانات|انشعابات کامل|انشعاب گاز",
    "has_water":          r"انشعاب آب|انشعابات کامل|فول امکانات",
    "has_electricity":    r"انشعاب برق|انشعابات کامل|فول امکانات",
    "has_parking":        r"پارکینگ",
    "has_warehouse":      r"انباری|انبار",
    "is_rebuilt":         r"بازسازی|نوسازی|صفر تا صد|کلید نخورده|شیک",
}
NEGATION_WORDS = r"بدون|فاقد|بی"

def build_negation_pattern(keyword_pattern):
    return rf"(?:{NEGATION_WORDS})[^.،؛\n]{{0,12}}(?:{keyword_pattern})"

_TRUE_TOKENS = {"true", "1", "yes", "بله", "دارد"}
_FALSE_TOKENS = {"false", "0", "no", "خیر", "ندارد"}

def _coerce_bool_scalar(x):
    if isinstance(x, bool):
        return x
    if x is None or x is pd.NA or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    t = str(x).strip().lower()
    if t in _TRUE_TOKENS: return True
    if t in _FALSE_TOKENS: return False
    return pd.NA

def to_nullable_bool(s):
    mapped = s.astype(object).map(_coerce_bool_scalar)
    mask = mapped.isna().to_numpy(dtype=bool)
    vals = mapped.fillna(False).astype(bool).to_numpy(dtype=bool)
    return pd.Series(pd.arrays.BooleanArray(vals, mask), index=s.index)

def impute_amenities(df, regex_map=AMENITY_REGEX, text_cols=("title", "description")):
    """Fill missing amenity flags from listing text, honoring negation."""
    out = df.copy()
    blob = out[list(text_cols)].astype("string").fillna("").agg(" ".join, axis=1)
    audit = {}
    for col, pattern in regex_map.items():
        if col not in out.columns:
            continue
        s = to_nullable_bool(out[col])
        na_before = int(s.isna().sum())
        na_mask = s.isna()
        neg_pattern = build_negation_pattern(pattern)
        text_hit = pd.Series(False, index=out.index)
        negated = pd.Series(False, index=out.index)
        if na_mask.any():
            text_hit.loc[na_mask] = blob.loc[na_mask].str.contains(pattern, regex=True, na=False).to_numpy(dtype=bool)
            negated.loc[na_mask] = blob.loc[na_mask].str.contains(neg_pattern, regex=True, na=False).to_numpy(dtype=bool)
        real_hit = text_hit & ~negated
        s = s.mask(na_mask & real_hit, True)
        s = s.mask(na_mask & negated, False)
        s = s.fillna(False).astype(bool)
        out[col] = s
        audit[col] = {"na_before": na_before, "recovered_true": int(real_hit.sum()),
                      "recovered_false_negated": int((negated & na_mask).sum()),
                      "true_rate_after": round(float(out[col].mean()), 4)}
    return out, pd.DataFrame(audit).T

df, amenity_audit = impute_amenities(df)
print(amenity_audit.to_string())

`has_gas` recovers the most from text - the word "gas" is common in listing text beyond just utility mentions, so this number is likely a bit generous. Negation catches a few thousand mentions across the amenity set (e.g. "بدون آسانسور" - no elevator), preventing them from being wrongly marked True.

### 2.4 Financial engineering: unified rahn/rent (@author: directive-specified formulas, applied by Mesbah's cleaning base)

Prices carry two problems: fake values (repeating digits, e.g. `111111`) and Rial-vs-Toman mixups (a price entered in Rial is 10x too large). Both get fixed into new `<col>_clean` columns, leaving the raw columns untouched.

Real estate ads come in different forms - sale price, deposit (rahn), or monthly rent - so most listings are missing two of the three. `unified_rahn` and `unified_rent` fill in whichever is missing using the standard market conversion rules, so every listing ends up with both features:

| From | To | Rule |
|---|---|---|
| rent | rahn | rent x 30 |
| rahn | rent | rahn / 30 |
| price | rahn | price / 5 |
| rahn | price | rahn x 5 |

In [ ]:
def is_repdigit(x):
    """True for junk like 1111, 11111, 99999 - all digits identical, 4+ long."""
    if pd.isna(x): return False
    s = str(int(x))
    return len(set(s)) == 1 and len(s) >= 4

def to_num(s):
    if s.dtype == object:
        s = s.map(fa_to_en_digits)
    s = s.astype("string").str.replace(",", "", regex=False).str.replace("٬", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def clean_financials(df, cols=("price_value", "credit_value", "rent_value"), price_floor=1_000_000):
    """Non-destructive: cleaned values go into new `<col>_clean` columns."""
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            continue
        x = to_num(out[col])
        rep = x.map(is_repdigit)
        x = x.where(~rep.fillna(False) & (x > 0))
        digit_len = x.dropna().astype("Int64").astype(str).str.len()
        rial_idx = digit_len[digit_len.isin([13, 14])].index
        x.loc[rial_idx] = x.loc[rial_idx] / 10   # Rial -> Toman
        x = x.where(x >= price_floor)
        out[f"{col}_clean"] = x
    return out

df = clean_financials(df)
print("price_value_clean coverage:", round(df['price_value_clean'].notna().mean(), 4))

In [ ]:
def add_unified_price_features(df, upper_pct=0.9999):
    """Derive unified_rahn / unified_rent from whichever of price/credit/rent
    is available. Capped at the 99.99th percentile - a handful of individual
    rent/credit values are still absurd (100+ trillion Toman/month has no
    real-world equivalent) and would otherwise dominate the combined feature."""
    out = df.copy()
    price, credit, rent = out["price_value_clean"], out["credit_value_clean"], out["rent_value_clean"]

    unified_rahn = credit.copy()
    unified_rahn = unified_rahn.where(unified_rahn.notna(), rent * 30)
    unified_rahn = unified_rahn.where(unified_rahn.notna(), price / 5)
    unified_rahn = unified_rahn.where(unified_rahn <= unified_rahn.quantile(upper_pct))

    unified_rent = rent.copy()
    unified_rent = unified_rent.where(unified_rent.notna(), credit / 30)
    unified_rent = unified_rent.where(unified_rent.notna(), unified_rahn / 30)
    unified_rent = unified_rent.where(unified_rent <= unified_rent.quantile(upper_pct))

    out["unified_rahn"] = unified_rahn
    out["unified_rent"] = unified_rent
    return out

df = add_unified_price_features(df)
print("unified_rahn coverage:", round(df['unified_rahn'].notna().mean(), 3))
print("unified_rent coverage:", round(df['unified_rent'].notna().mean(), 3))
print(df[["unified_rahn", "unified_rent"]].describe())

### 2.5 Inflation adjustment (@author: directive-specified formula)

Nominal prices rise over time partly from inflation, not real value growth. `inflation_adjusted_value` re-expresses `unified_rahn` in today's terms, assuming 3.5% monthly inflation: `value * 1.035^months_since_listing`. "Present" is set to the latest month seen in the data (keeps the feature inside the dataset's own timeframe).

In [ ]:
def add_inflation_adjustment(df, value_col="unified_rahn", date_col="created_at_month"):
    out = df.copy()
    dt = pd.to_datetime(out[date_col], errors="coerce")
    present = dt.max()
    months_since = ((present.year - dt.dt.year) * 12 + (present.month - dt.dt.month)).clip(lower=0)
    out["months_since_listing"] = months_since
    out[f"{value_col}_inflation_adjusted"] = out[value_col] * (1.035 ** months_since)
    return out

df = add_inflation_adjustment(df)
print("mean unified_rahn (nominal):", round(df["unified_rahn"].mean(), 0))
print("mean unified_rahn (inflation-adjusted):", round(df["unified_rahn_inflation_adjusted"].mean(), 0))

### 2.6 Grouped imputation (@author: directive-specified rule)

Remaining gaps in `unified_rahn` are filled with the group mean, grouped by `city_slug`, `neighborhood_slug`, and property type. `property_type` itself is ~97% empty, so `cat3_slug` (apartment/house/shop/...) is used instead - it captures the same idea and is essentially complete. Falls back to a coarser grouping when a specific (city, neighborhood, type) combination has no data of its own.

In [ ]:
def impute_grouped(df, col, group_levels=(("city_slug", "neighborhood_slug", "cat3_slug"),
                                           ("city_slug", "cat3_slug"),
                                           ("cat3_slug",))):
    """Mean (numeric) or mode (categorical) imputation, grouped by location + property type,
    falling back to a broader group when the specific combination is empty."""
    out = df.copy()
    is_numeric = pd.api.types.is_numeric_dtype(out[col])
    s = out[col].copy()
    for group_cols in group_levels:
        if is_numeric:
            filler = out.groupby(list(group_cols))[col].transform("mean")
        else:
            filler = out.groupby(list(group_cols))[col].transform(lambda g: g.mode().iloc[0] if not g.mode().empty else np.nan)
        s = s.fillna(filler)
    global_fill = s.mean() if is_numeric else (s.mode().iloc[0] if not s.mode().empty else np.nan)
    s = s.fillna(global_fill)
    out[col] = s
    return out

before_na = df["unified_rahn"].isna().sum()
df = impute_grouped(df, "unified_rahn")
print(f"unified_rahn missing before: {before_na:,}  after grouped imputation: {df['unified_rahn'].isna().sum():,}")

### 2.7 City type, house age, residential filter (@author: Mesbah)

In [ ]:
cls = pd.read_csv("iran_city_classification.csv")
cls.columns = ["city_slug", "city_type_fa"]
cls["city_type"] = np.where(cls["city_type_fa"].str.contains("کلان"), "Metropolis", "Small City")
df = df.merge(cls[["city_slug", "city_type"]], on="city_slug", how="left")

old = (df["construction_year"] < 1396)
df["is_old_house"] = old.astype("boolean")
df.loc[df["construction_year"].isna(), "is_old_house"] = pd.NA

RES_CAT2 = ["residential-sell", "residential-rent"]
EXCLUDE_CAT3 = {"plot-old", "presell"}
res = df[df["cat2_slug"].isin(RES_CAT2) & ~df["cat3_slug"].isin(EXCLUDE_CAT3)].copy()
print(f"residential frame: {len(res):,} rows ({len(res)/len(df):.1%} of all)")
df["price_toman"] = df["price_value_clean"]
res["price_toman"] = res["price_value_clean"]

## 3. Descriptive statistics

### Q1 - Ad distribution by category (@author: Mahya)

Counting listings by level-2 and level-3 category, showing the top 15 of each to keep the chart readable.

In [ ]:
def get_top_n_categories(series, n=15):
    return series.value_counts().nlargest(n).index

top_cat2 = get_top_n_categories(df["cat2_slug"])
top_cat3 = get_top_n_categories(df["cat3_slug"])
df_cat2_plot = df[df["cat2_slug"].isin(top_cat2)]
df_cat3_plot = df[df["cat3_slug"].isin(top_cat3)]

fig, axes = plt.subplots(2, 1, figsize=(14, 12))
sns.countplot(data=df_cat2_plot, y="cat2_slug", ax=axes[0], palette="viridis",
              order=df_cat2_plot["cat2_slug"].value_counts().index)
axes[0].set_title("Ads by Category Level 2")
axes[0].set_xlabel("Number of Ads"); axes[0].set_ylabel("cat2_slug")

sns.countplot(data=df_cat3_plot, y="cat3_slug", ax=axes[1], palette="magma",
              order=df_cat3_plot["cat3_slug"].value_counts().index)
axes[1].set_title("Ads by Category Level 3 (Top 15)")
axes[1].set_xlabel("Number of Ads"); axes[1].set_ylabel("cat3_slug")
plt.tight_layout()
plt.show()

Residential sale and rent dominate the platform - apartment-sell and apartment-rent alone make up more than half of all listings. Commercial and industrial categories are a small share by comparison.

### Q2 - Construction year histogram (@author: Mahya)

Using the cleaned `construction_year` from section 2.2. Listings concentrate in the last ~15 years, which fits an actively-listed platform (older buildings are less likely to be re-listed for sale/rent).

In [ ]:
plt.figure(figsize=(13, 6))
sns.histplot(df["construction_year"].dropna(), discrete=True, color="teal")
plt.title("Construction Year Distribution")
plt.xlabel("year (Shamsi)"); plt.ylabel("number of listings")
plt.tight_layout()
plt.show()

### Q3 - Are there seasonal spikes in listing volume? (@author: Mahya)

Two ways to look at this: a straight month-by-month timeline, or averaging each Persian calendar month (Farvardin, Ordibehesht, ...) across all years to see if a particular time of year is consistently busier. The question asks about "certain times of the year", so the month-name average is the more direct answer - used here.

In [ ]:
d = df.copy()
d["created_at_month"] = pd.to_datetime(d["created_at_month"], errors="coerce")
d = d.dropna(subset=["created_at_month"])

def categorize_market(slug):
    slug = str(slug).lower()
    if "sell" in slug: return "Sale"
    if "rent" in slug: return "Rent"
    return "Other"

d["market_type"] = d["cat2_slug"].apply(categorize_market)
d = d[d["market_type"].isin(["Sale", "Rent"])].copy()

persian_months = {1:"Farvardin",2:"Ordibehesht",3:"Khordad",4:"Tir",5:"Mordad",6:"Shahrivar",
                   7:"Mehr",8:"Aban",9:"Azar",10:"Dey",11:"Bahman",12:"Esfand"}

def get_persian_month(dt):
    try:
        j = jdatetime.date.fromgregorian(day=dt.day, month=dt.month, year=dt.year)
        return persian_months.get(j.month)
    except Exception:
        return None

d["persian_month"] = d["created_at_month"].apply(get_persian_month)
d = d.dropna(subset=["persian_month"])
d["year_month_temp"] = d["created_at_month"].dt.to_period("M")

monthly_counts = d.groupby(["year_month_temp", "persian_month", "market_type"], observed=True).size().reset_index(name="count")
seasonal = monthly_counts.groupby(["persian_month", "market_type"], observed=True)["count"].mean().reset_index()
month_order = list(persian_months.values())
seasonal["persian_month"] = pd.Categorical(seasonal["persian_month"], categories=month_order, ordered=True)
seasonal = seasonal.sort_values("persian_month")

plt.figure(figsize=(13, 6))
sns.lineplot(data=seasonal, x="persian_month", y="count", hue="market_type", marker="o", linewidth=3,
             palette={"Sale": "#27ae60", "Rent": "#2980b9"})
plt.title("Average Monthly Ad Volume by Persian Calendar Month")
plt.xlabel("month"); plt.ylabel("average number of ads")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for m in ["Sale", "Rent"]:
    td = seasonal[seasonal["market_type"] == m]
    top = td.loc[td["count"].idxmax()]
    low = td.loc[td["count"].idxmin()]
    print(f"{m}: peak={top['persian_month']} ({top['count']:.0f} ads), low={low['persian_month']} ({low['count']:.0f} ads)")

The Farvardin/Dey/Bahman/Esfand months show much lower averages, but this is most likely a data-coverage artifact rather than a real seasonal dip: those months sit at the edges of the collection window and are only backed by one partial year of data each, while months like Tir or Mehr are averaged across multiple full years. The mid-year months (Ordibehesht through Azar) give the more trustworthy read, and within that range volume is fairly stable for both sale and rent, without one dramatic spike.

### Q4 - Sale price distribution by category (@author: Erfan)

Comparing `price_toman` across level-3 categories with a boxplot. Real estate prices mix genuine high-value outliers (a huge villa or commercial complex) with fake ones (typos, repeated-digit placeholders) - already filtered out in `price_value_clean`. What is left here is trimming the plot itself: Tukey's rule with a wider 3x IQR multiplier per category, so only the most extreme values are dropped and normal high-end listings still show as dots.

In [ ]:
df_sell = df.dropna(subset=["price_toman"]).copy()
print("rows with a usable price_toman:", len(df_sell))

def filter_extreme_outliers(df, cat_col, value_col, k=3.0):
    parts = []
    for cat, g in df.groupby(cat_col, observed=True):
        q1, q3 = g[value_col].quantile([0.25, 0.75])
        iqr = q3 - q1
        parts.append(g[g[value_col] <= q3 + k * iqr])
    return pd.concat(parts)

before = len(df_sell)
df_q4 = filter_extreme_outliers(df_sell, "cat3_slug", "price_toman")
print(f"dropped {before - len(df_q4):,} extreme outliers (3x IQR per category)")

plt.figure(figsize=(16, 8))
sns.boxplot(data=df_q4, x="cat3_slug", y="price_toman", whis=1.5, showfliers=True, palette="viridis",
            fliersize=3, flierprops={"marker": "o", "markerfacecolor": "orange", "markeredgecolor": "red", "alpha": 0.4})
plt.title("Sale Price by Category (extreme outliers removed, 3x IQR)")
plt.xlabel("category"); plt.ylabel("price (Toman)")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x/1e9:.1f}B"))
plt.tight_layout()
plt.show()

Commercial categories (shop, office) show a higher median than residential ones despite fewer listings - commercial space tends to cost more per square meter. `plot-old` (land only) has the widest spread relative to its median, since land value depends heavily on location and zoning with no building quality to average things out.

### Q5 - Geographic distribution of listings (@author: Erfan)

Two visualizations, both after the same cleanup: dropping missing coordinates and filtering out points outside Iran using a point-in-polygon check against the country's real border shape (not just a bounding box).

In [ ]:
geojson_path = "IRN.geo.json"
url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries/IRN.geo.json"
if not os.path.exists(geojson_path):
    import urllib.request
    urllib.request.urlretrieve(url, geojson_path)
with open(geojson_path, "r", encoding="utf-8") as f:
    iran_geojson = js.load(f)

iran_paths = []
for feature in iran_geojson["features"]:
    geom = feature["geometry"]
    if geom["type"] == "Polygon":
        for sub in geom["coordinates"]:
            iran_paths.append(Path(sub))
    elif geom["type"] == "MultiPolygon":
        for poly in geom["coordinates"]:
            for sub in poly:
                iran_paths.append(Path(sub))

df_geo = df.dropna(subset=["latitude", "longitude"]).copy()
coords = df_geo[["longitude", "latitude"]].values
is_inside = np.zeros(len(coords), dtype=bool)
for path in iran_paths:
    is_inside |= path.contains_points(coords)
df_iran = df_geo[is_inside]
print(f"valid coordinates: {len(df_geo):,}; strictly inside Iran: {len(df_iran):,} ({len(df_iran)/len(df_geo):.1%})")

**Visualization 1 - Folium interactive heatmap.**

In [ ]:
iran_map = folium.Map(location=[32.4279, 53.6880], zoom_start=5, tiles="CartoDB positron")
heat_data = df_iran[["latitude", "longitude"]].values.tolist()
custom_gradient = {0.2: "blue", 0.4: "cyan", 0.6: "lime", 0.8: "yellow", 0.95: "orange", 1.0: "red"}
HeatMap(heat_data, radius=8, blur=5, max_zoom=12, min_opacity=0.33, gradient=custom_gradient).add_to(iran_map)
iran_map.save("iran_real_estate_heatmap.html")
iran_map

**Visualization 2 - static hexbin with border overlay and log color scale** (Tehran's volume is large enough to drown out every other city on a linear scale).

In [ ]:
regional_centers = {
    "Tehran/Alborz": (51.3892, 35.6892), "Mashhad": (59.6159, 36.2972),
    "Isfahan": (51.6660, 32.6546), "Shiraz": (52.5388, 29.5918),
    "Tabriz": (46.2919, 38.0962), "Ahvaz": (48.6706, 31.3183),
    "Rasht": (49.5831, 37.2808), "Kermanshah": (47.0778, 34.3142),
    "Kerman": (57.0740, 30.2839), "Zahedan": (60.8629, 29.4963),
}
plt.figure(figsize=(13, 10))
for path in iran_paths:
    v = path.vertices
    plt.plot(v[:, 0], v[:, 1], color="black", linewidth=1.2, zorder=2)

hb = plt.hexbin(df_iran["longitude"], df_iran["latitude"], gridsize=130, cmap="YlOrRd",
                bins="log", mincnt=1, edgecolors="none", alpha=0.85, zorder=1)
for city, (lon, lat) in regional_centers.items():
    plt.scatter(lon, lat, color="black", marker="o", s=25, zorder=3)
    plt.text(lon + 0.15, lat + 0.15, city, fontsize=8, fontweight="bold", zorder=4,
              bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.5))
cb = plt.colorbar(hb, orientation="vertical", fraction=0.03, pad=0.02)
cb.set_label("listing density (log10 scale)")
plt.title("Listing Density (log scale, Iran border only)")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.xlim(43.5, 64.0); plt.ylim(24.5, 40.5)
plt.tight_layout()
plt.show()

Tehran/Alborz is the clear center of gravity. The log scale keeps Mashhad, Isfahan, the Caspian coast and southern hubs (Fars/Khuzestan) visible instead of one saturated blob - this roughly matches Iran's population and economic centers.

### Q6 - Monthly average rent trend (@author: Abbas)

`effective_rent` combines the monthly rent with a small share of the deposit (`rent + 3% x credit`), since many rental ads mix the two. After IQR filtering and dropping unrealistically tiny values, months are converted to the Jalali calendar for readable labels. Coverage before Ordibehesht 1403 is too sparse (single-digit to low-hundreds of listings per month) to trust, so the plot is restricted to the reliable window.

In [ ]:
res_rent = df[df["cat2_slug"] == "residential-rent"].copy()
res_rent["effective_rent"] = res_rent["rent_value_clean"].fillna(0) + 0.03 * res_rent["credit_value_clean"].fillna(0)
res_rent = res_rent[res_rent["effective_rent"] > 0]

q1, q3 = res_rent["effective_rent"].quantile([0.25, 0.75])
iqr = q3 - q1
res_rent_clean = res_rent[(res_rent["effective_rent"] >= q1 - 1.5*iqr) & (res_rent["effective_rent"] <= q3 + 1.5*iqr)]
res_rent_clean = res_rent_clean[res_rent_clean["effective_rent"] >= 1_000_000].copy()

res_rent_clean["created_at_month"] = pd.to_datetime(res_rent_clean["created_at_month"], errors="coerce")
res_rent_clean = res_rent_clean.dropna(subset=["created_at_month"])
res_rent_clean["jalali_month"] = res_rent_clean["created_at_month"].apply(
    lambda x: jdatetime.date.fromgregorian(date=x).strftime("%Y-%m"))

monthly_rent = res_rent_clean.groupby("jalali_month")["effective_rent"].mean().reset_index().sort_values("jalali_month")
monthly_rent_plot = monthly_rent[(monthly_rent["jalali_month"] >= "1403-02") & (monthly_rent["jalali_month"] <= "1403-09")].copy()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(data=monthly_rent_plot, x="jalali_month", y="effective_rent", marker="o")
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
plt.title("Trend of Average Monthly Rent (reliable window: 1403-02 to 1403-09)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Average effective rent is fairly flat across the reliable months, in the 16-17 million Toman range, with no runaway trend inside this short window.

### Q7 - Nominal vs. real (inflation-adjusted) price trend, 1400-1403 (@author: Abbas)

Nominal prices rise over time partly from inflation, not real value growth. This uses annually-measured CPI figures (base year 1403) to compute a real price, separate from the generic `inflation_adjusted_value` feature in section 2.5 - actual yearly CPI is more accurate for a specific historical trend like this one than a flat monthly assumption.

In [ ]:
res_sel = df.loc[df["cat2_slug"] == "residential-sell", ["created_at_month", "price_value"]].copy()
res_sel["created_at_month"] = pd.to_datetime(res_sel["created_at_month"], errors="coerce")
res_sel["year_shamsi"] = res_sel["created_at_month"].apply(
    lambda x: jdatetime.date.fromgregorian(date=x.date()).year if pd.notnull(x) else None)
res_sel = res_sel[res_sel["year_shamsi"].isin([1400, 1401, 1402, 1403])]
print(res_sel["year_shamsi"].value_counts().sort_index())

cpi = {1400: 100.0, 1401: 153.1, 1402: 225.6, 1403: 306.4}
res_sel["real_price"] = res_sel["price_value"] * (cpi[1403] / res_sel["year_shamsi"].map(cpi))
trend_df = res_sel.groupby("year_shamsi")[["price_value", "real_price"]].mean().reset_index()
trend_df.columns = ["Year", "Mean Nominal Price", "Mean Real Price (1403 Base)"]
print(trend_df)

plt.figure(figsize=(9, 5))
years = trend_df["Year"].astype(str)
plt.plot(years, trend_df["Mean Nominal Price"]/1e9, marker="o", color="#d9534f", label="Nominal Price")
plt.plot(years, trend_df["Mean Real Price (1403 Base)"]/1e9, marker="s", color="#5cb85c", label="Real Price (Base: 1403)")
plt.title("Housing Price Trend (1400-1403)")
plt.xlabel("Year (Shamsi)"); plt.ylabel("Avg Price (Billion Toman)")
plt.legend()
plt.tight_layout()
plt.show()

1400 and 1401 have very few listings (5 and 53 respectively) - not enough to treat those two averages as reliable, and the sharp jump/drop between them is more likely a small-sample artifact than a real market swing. The 1402-to-1403 comparison rests on a much larger sample and is more trustworthy: nominal price rises from about 7.5B to 16.9B Toman, and even after removing the inflation effect, the real price still climbs from about 10.2B to 16.9B - so part of this period's price growth reflects an actual increase in property value, not just inflation.

### Q8 - Correlation matrix: price, size, rooms, location (@author: Mesbah)

Spearman rank correlation, since price and size are both right-skewed and `rooms_count` behaves more like an ordinal scale than a true continuous number. `regular_person_capacity` (mentioned in the assignment) turns out to be a daily-rent-only field, essentially empty for regular residential listings, so `rooms_count` is used instead. Raw latitude/longitude carry little price information on their own, so `Distance_to_City_Center` (distance to each city's own median coordinate) is used in their place.

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

q8 = res[res["cat2_slug"] == "residential-sell"].copy()
q8["Price"] = q8["price_toman"]
cap = pd.to_numeric(q8.get("regular_person_capacity"), errors="coerce")
print("regular_person_capacity coverage:", cap.notna().mean() if cap is not None and len(cap) else "column dropped earlier - confirms it is unusable here")

cols = ["Price", "land_size", "building_size", "rooms_count", "latitude", "longitude"]
M = q8[cols].apply(pd.to_numeric, errors="coerce")
for c in ["Price", "land_size", "building_size"]:
    M.loc[M[c] > M[c].quantile(0.99), c] = np.nan

corr = M.corr(method="spearman")
print(corr.round(3).to_string())

dist = squareform(1 - corr.abs().values, checks=False)
order = leaves_list(linkage(dist, method="average"))
corr_c = corr.iloc[order, order]
plt.figure(figsize=(7.5, 6))
sns.heatmap(corr_c, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={"label": "Spearman rho"})
plt.title("Correlation Matrix (residential-sell)")
plt.tight_layout(); plt.savefig("q8_spearman_heatmap.png", dpi=130); plt.show()

METROPOLISES = ["tehran", "mashhad", "isfahan", "karaj", "shiraz", "tabriz"]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

q8m = q8[q8["city_slug"].isin(METROPOLISES)].dropna(subset=["latitude", "longitude"]).copy()
centers = q8m.groupby("city_slug")[["latitude", "longitude"]].median()
q8m = q8m.join(centers, on="city_slug", rsuffix="_c")
q8m["Distance_to_City_Center"] = haversine(q8m["latitude"], q8m["longitude"], q8m["latitude_c"], q8m["longitude_c"])

rows = []
for city, gg in q8m.groupby("city_slug"):
    d = gg.dropna(subset=["Price", "Distance_to_City_Center"])
    d = d[d["Price"] <= d["Price"].quantile(0.99)]
    if len(d) >= 100:
        rho, pv = stats.spearmanr(d["Distance_to_City_Center"], d["Price"])
        rows.append((city, len(d), round(rho, 3), f"{pv:.1e}"))
print(pd.DataFrame(rows, columns=["city", "n", "rho", "p_value"]).to_string(index=False))

`building_size` and `rooms_count` move together the most (rho ~0.75), and both drive price up (rho ~0.5 and ~0.47) - floor area looks like the main price driver, with room count mostly a proxy for it. Distance-to-center is negative in every metro (further out = cheaper), strongest in Isfahan/Mashhad (rho ~-0.39) and weakest in Tehran (rho ~-0.04) - Tehran likely has multiple high-value sub-centers (e.g. the north), so a single centroid does not capture its price structure well.

### Q9 - Amenity concentration by neighborhood (@author: Mesbah)

Grouping by `neighborhood_slug` and computing percentage density for 5 amenities, not raw counts, so small and large neighborhoods are directly comparable. Neighborhoods under 15 listings are dropped - too few listings to trust the percentage.

In [ ]:
Q9_AMENITIES = ["has_balcony", "has_elevator", "has_security_guard", "has_barbecue", "has_pool"]
g = res.groupby("neighborhood_slug")
dens = g[Q9_AMENITIES].mean()
dens["n_listings"] = g.size()
dens = dens[dens["n_listings"] >= 15]
top = dens.sort_values("n_listings", ascending=False).head(30)
print(f"neighborhoods with >= 15 listings: {len(dens):,}")

plt.figure(figsize=(8, 11))
sns.heatmap(top[Q9_AMENITIES] * 100, annot=True, fmt=".0f", cmap="YlOrRd", vmin=0, vmax=100,
            linewidths=.5, cbar_kws={"label": "% of listings with amenity"})
plt.title("Amenity Density by Neighborhood (Top 30 by volume)")
plt.tight_layout(); plt.savefig("q9_amenity_density.png", dpi=130); plt.show()

for a in Q9_AMENITIES:
    t = dens.sort_values(a, ascending=False).head(3)
    print(f"  {a:20s}:", ", ".join(f"{i} ({v:.0%})" for i, v in t[a].items()))

Dense high-rise neighborhoods sit near 100% on `has_elevator`, while lower-rise/villa-style neighborhoods spike instead on `has_pool` and `has_barbecue`. `has_security_guard` tops out lower overall and concentrates in pricier neighborhoods - a guard is a staffing cost, so it appears more as a premium feature than a baseline one.

## 4. Hypothesis testing

H1 and H2 (@author: Mesbah) compare `building_size` between groups; H3 (@author: Erfan) and H4 (@author: Abbas) compare price between groups. All four reuse the same toolkit below.

Group sizes here run into the hundreds of thousands, where a plain Shapiro-Wilk test at alpha=0.05 rejects normality almost regardless of the data - the test has too much power at this scale. Alpha is instead scaled down as the group grows, giving a stricter bar for what counts as "not normal" before falling back to a non-parametric test.

In [ ]:
def adaptive_alpha(n):
    if n < 1_000: return 0.05
    elif n < 10_000: return 0.01
    elif n < 100_000: return 0.001
    else: return 0.0001

def shapiro_check(series, cap=5000, seed=0):
    """Shapiro-Wilk on a capped sample (unreliable/slow well before n=100k per scipy's own docs).
    The group's real size still drives adaptive_alpha above."""
    n_true = len(series)
    rng = np.random.default_rng(seed)
    vals = series.to_numpy()
    if n_true > cap:
        vals = rng.choice(vals, size=cap, replace=False)
    stat, p = stats.shapiro(vals)
    return stat, p, n_true

def cohens_d(a, b):
    na, nb = len(a), len(b)
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    pooled_sd = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_sd

def welch_t(a, b, alpha=0.05):
    """Welch's t-test for the mean difference + analytical 95% CI + Cohen's d."""
    res_t = stats.ttest_ind(a, b, equal_var=False)
    na, nb = len(a), len(b)
    ma, mb = np.mean(a), np.mean(b)
    va, vb = np.var(a, ddof=1), np.var(b, ddof=1)
    se = np.sqrt(va / na + vb / nb)
    dof = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    tcrit = stats.t.ppf(1 - alpha / 2, dof)
    diff = ma - mb
    return {"t": res_t.statistic, "p": res_t.pvalue, "mean_diff": diff,
            "ci95": (diff - tcrit * se, diff + tcrit * se), "cohens_d": cohens_d(a, b)}

def mannwhitney(a, b, alternative="two-sided"):
    """Rank-based test: whether a random draw from a tends to rank above one from b."""
    U, p = stats.mannwhitneyu(a, b, alternative=alternative)
    rbc = 2 * U / (len(a) * len(b)) - 1
    return {"U": U, "p": p, "rank_biserial": rbc}

def run_two_group_test(a, b, label_a, label_b, mwu_alternative, n_strat=10000, seed=0):
    print(f"n {label_a}={len(a):,}   n {label_b}={len(b):,}")
    print(f"mean {label_a}={a.mean():.1f}  mean {label_b}={b.mean():.1f}  diff={a.mean()-b.mean():+.1f}")
    print(f"median {label_a}={a.median():.0f}  median {label_b}={b.median():.0f}")

    alpha = adaptive_alpha(len(a))
    _, p_norm, n_true = shapiro_check(a)
    print(f"Shapiro p={p_norm:.2e} vs adaptive alpha={alpha} (n={n_true:,}) -> "
          f"{'normal enough' if p_norm > alpha else 'not normal'}")

    w = welch_t(a.values, b.values)
    print(f"Welch's t: t={w['t']:.2f}  p={w['p']:.2e}  mean_diff={w['mean_diff']:+.2f}  "
          f"95% CI=({w['ci95'][0]:.2f}, {w['ci95'][1]:.2f})  Cohen's d={w['cohens_d']:+.4f}")
    mw = mannwhitney(a.values, b.values, alternative=mwu_alternative)
    print(f"Mann-Whitney U ({mwu_alternative}): p={mw['p']:.2e}  rank-biserial={mw['rank_biserial']:+.4f}")

    rng = np.random.default_rng(seed)
    a_s = pd.Series(rng.choice(a.values, min(n_strat, len(a)), replace=False))
    b_s = pd.Series(rng.choice(b.values, min(n_strat, len(b)), replace=False))
    w_s = welch_t(a_s.values, b_s.values)
    print(f"[downsample n={n_strat}/group] Welch p={w_s['p']:.2e}  Cohen's d={w_s['cohens_d']:+.4f}")
    return w, mw, w_s

### Anomaly check: `building_size` before H1/H2 (@author: Mesbah)

Before dropping missing/odd `building_size` values, checking where they occur. The 99.9th percentile of `building_size` jumps from ~950 sqm straight to 100,000+ sqm - not a real house. Many of the values up there are round numbers (100000, 120000) or repeated-digit junk (11111), the same kind of scraper error already handled for prices. Flagging both missing and implausible (repeated-digit, or over 3,000 sqm) before deciding what to drop.

In [ ]:
autopsy = res.copy()
autopsy["is_missing_size"] = autopsy["building_size"].isna()
rep = autopsy["building_size"].map(is_repdigit)
too_big = autopsy["building_size"] > 3000
autopsy["is_implausible_size"] = (rep.fillna(False) | too_big.fillna(False)) & ~autopsy["is_missing_size"]
print(f"missing: {autopsy['is_missing_size'].sum():,} ({autopsy['is_missing_size'].mean():.3%})")
print(f"implausible: {autopsy['is_implausible_size'].sum():,} ({autopsy['is_implausible_size'].mean():.3%})")
print(autopsy.groupby("user_type", observed=True)[["is_missing_size", "is_implausible_size"]].mean().to_string())

clean_size = autopsy[~autopsy["is_missing_size"] & ~autopsy["is_implausible_size"]].copy()
print(f"clean rows kept: {len(clean_size):,} ({len(clean_size)/len(autopsy):.2%} of res)")

Individual sellers post junk `building_size` values noticeably more often than agencies do, and houses/villas are messier than apartments (likely mixing up land size and building size). Almost all rows survive both filters.

### Hypothesis 1 - Are metro homes smaller than small-city homes? (@author: Mesbah)

H1: average `building_size` in Metropolises is smaller than in Small Cities (migration pressure pushing people into more compact homes).

In [ ]:
metro = clean_size.loc[clean_size["city_type"] == "Metropolis", "building_size"]
small = clean_size.loc[clean_size["city_type"] == "Small City", "building_size"]
w1, mw1, w1_down = run_two_group_test(metro, small, "metro", "small", mwu_alternative="less")

The p-value is tiny, but Cohen's d is only about -0.02, far below even a small effect (0.2), and medians are identical (100 vs 100). At a normal sample size (10,000/group) the Welch p-value is no longer significant. The apparent effect at full scale is n doing the work, not a real size difference: metro and small-city homes come out essentially the same size. If migration pressure is real, it more likely shows up as a higher price per square meter rather than smaller homes.

### Hypothesis 2 - "Old houses were more spacious" (@author: Mesbah)

H2: average `building_size` of houses built before 1396 is larger than newer houses - testing a common piece of folk wisdom against the listings.

In [ ]:
old = clean_size.loc[clean_size["is_old_house"] == True, "building_size"]
new = clean_size.loc[clean_size["is_old_house"] == False, "building_size"]
w2, mw2, w2_down = run_two_group_test(old, new, "old", "new", mwu_alternative="greater")

Old houses measure about 20 sqm smaller on average, not bigger, and Cohen's d (~0.20) is a real, if modest, effect - unlike H1, this holds up at a downsampled n=10,000/group. The nostalgia does not hold up in current listings: new construction is built and marketed on floor area, while what remains listed as pre-1396 skews toward smaller, older inner-city apartments. Traditional courtyard houses/villas, which the nostalgia likely refers to, have mostly left the current for-sale/for-rent pool.

### Hypothesis 3 - Does a business deed raise commercial sale price? (@author: Erfan)

A business deed (`has_business_deed`) means legally-recognized commercial ownership. The assignment asks about the effect/influence of the deed, which is a directional question, so only the one-sided test (deed -> higher price) is used here.

H3: average sale price of commercial listings with a business deed is higher than those without.

In [ ]:
commercial_cats = ["shop-sell", "office-sell", "industry-agriculture-business-sell"]
df_comm = df[df["cat3_slug"].isin(commercial_cats)].dropna(subset=["price_toman"]).copy()
df_comm["has_business_deed"] = df_comm["has_business_deed"].fillna(False).astype(bool)
print(f"commercial listings with a usable price: {len(df_comm):,}")
print(df_comm["has_business_deed"].value_counts())

with_deed = df_comm.loc[df_comm["has_business_deed"], "price_toman"]
without_deed = df_comm.loc[~df_comm["has_business_deed"], "price_toman"]
w31, mw31, w31_down = run_two_group_test(with_deed, without_deed, "with_deed", "without_deed", mwu_alternative='two-sided')
print('##############')
w32, mw32, w32_down = run_two_group_test(with_deed, without_deed, "with_deed", "without_deed", mwu_alternative="greater")

Listings with a business deed average about 2.4 billion Toman higher, and the one-sided Mann-Whitney test confirms the direction (p well below 0.001). Cohen's d (~0.04) and rank-biserial (~0.07) are small but non-zero, and the effect survives the downsample check. A deed is associated with a real, if modest, price premium - one factor among several (location, size, condition) rather than the dominant driver.

### Hypothesis 4 - Luxury vs. non-luxury amenities and price (@author: Abbas)

Luxury amenities: pool, jacuzzi, sauna, barbecue. Non-luxury/basic amenities: elevator, warehouse, parking. H4 (part 1): listings with at least one luxury amenity average a higher price. H4 (part 2): does the same hold for the non-luxury set?

Price is log-transformed first (right-skewed), then the same IQR filtering and toolkit as H1-H3 is applied for consistency, using the negation-aware amenity flags from section 2.3 rather than the raw partially-missing booleans.

In [ ]:
df_clean = df[df["price_value"].notna() & (df["price_value"] > 0)].copy()
q1, q3 = df_clean["price_value"].quantile([0.25, 0.75])
iqr = q3 - q1
df_no_out = df_clean[(df_clean["price_value"] >= q1 - 1.5*iqr) & (df_clean["price_value"] <= q3 + 1.5*iqr)].copy()
df_no_out["log_price"] = np.log(df_no_out["price_value"])
print("rows after IQR filter:", len(df_no_out))

lux_cols = ["has_pool", "has_jacuzzi", "has_sauna", "has_barbecue"]
lux_mask = df_no_out[lux_cols].any(axis=1)
non_lux_mask = ~df_no_out[lux_cols].any(axis=1)
lux_log = df_no_out.loc[lux_mask, "log_price"].dropna()
non_lux_log = df_no_out.loc[non_lux_mask, "log_price"].dropna()

print("--- luxury vs non-luxury ---")
w4a, mw4a, w4a_down = run_two_group_test(lux_log, non_lux_log, "luxury", "non_luxury", mwu_alternative="greater")

In [ ]:
basic_cols = ["has_elevator", "has_warehouse", "has_parking"]
basic_mask = df_no_out[basic_cols].any(axis=1)
without_basic_mask = ~df_no_out[basic_cols].any(axis=1)
basic_log = df_no_out.loc[basic_mask, "log_price"].dropna()
without_basic_log = df_no_out.loc[without_basic_mask, "log_price"].dropna()

print("--- basic (non-luxury) vs without ---")
w4b, mw4b, w4b_down = run_two_group_test(basic_log, without_basic_log, "has_basic", "without_basic", mwu_alternative="greater")

Both comparisons are significant in the expected direction, but the basic-amenity effect is considerably stronger than the luxury one. This fits a simple story: elevator/parking are close to a baseline expectation in this market, so their absence hurts value sharply, while pool/sauna/jacuzzi/barbecue are a smaller premium relevant mostly to a narrower luxury segment. Non-luxury amenities look like the more consistent price driver of the two.

## 3. Machine Learning - Price Prediction (Problem 2)

The goal of this section is to predict a unified, inflation-adjusted property value for both rent (Rahn) and sale listings, from property attributes only. The target is `unified_rahn_inflation_adjusted` built in Section 2.

A few design choices are made to keep the result honest:

- **No target leakage.** The raw money columns (`price_value`, `rent_value`, `credit_value`, the `transformable_*`/`transformed_*` columns, and the `unified_*` columns the target is derived from) are removed from the input features. Only physical, location, and categorical attributes are used.
- **Split before anything else.** An 85/15 train/test split is made first, with a fixed seed. Every statistic used for imputation and every encoder is fit on the training part only, then applied to the test part.
- **Log target.** Property values are heavily right-skewed and span rent (millions) to sale (billions), so the model is trained on `log1p(value)`. Model quality is therefore reported as R2 on the log scale (the standard choice for skewed monetary targets), together with MAE, RMSE, and a median absolute percentage error. The original-scale R2 is also shown for transparency, but it is naturally low because a few billion-Toman sale errors dominate that metric while log space treats relative errors evenly.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import category_encoders as ce
import lightgbm as lgb
import xgboost as xgb
import joblib, os, json as _json, hashlib

os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
ML_SEED = 42
TARGET = "unified_rahn_inflation_adjusted"
np.random.seed(ML_SEED)

### Phase 1 - Leakage-safe feature frame and split

A deep copy of the Section-2 dataframe is taken so the ML work does not touch the statistical-analysis data. Features are selected by an explicit allow-list (physical + location + category attributes), which guarantees no money column leaks in. Rows without a target value are dropped. A few simple engineered features are added (building age, area per room, floor ratio, sell/commercial flags, log sizes, and a city x category key), none of which use the target.

In [ ]:
import gc
# --- free memory held by the statistical sections before the ML step ---
for _v in ["res","df_cat2_plot","df_cat3_plot","top_cat2","top_cat3","d","monthly_counts","seasonal",
           "df_sell","df_q4","df_geo","coords","is_inside","df_iran","iran_map","heat_data","iran_paths",
           "res_rent","res_rent_clean","monthly_rent","monthly_rent_plot","res_sel","trend_df",
           "q8","M","corr","corr_c","q8m","centers","dens","top","g","amenity_audit",
           "autopsy","clean_size","metro","small","old","new","df_comm","with_deed","without_deed",
           "df_clean","df_no_out","lux_log","non_lux_log","basic_log","without_basic_log",
           "lux_mask","non_lux_mask","basic_mask","without_basic_mask"]:
    globals().pop(_v, None)
gc.collect()

ML_FEATURES = ["cat2_slug","cat3_slug","city_slug","neighborhood_slug","user_type","land_size",
    "building_size","deed_type","has_business_deed","floor","rooms_count","total_floors_count",
    "unit_per_floor","has_balcony","has_elevator","has_warehouse","has_parking","construction_year",
    "is_rebuilt","has_water","has_warm_water_provider","has_electricity","has_gas","has_heating_system",
    "has_cooling_system","restroom_type","has_security_guard","has_barbecue","building_direction",
    "has_pool","has_jacuzzi","has_sauna","regular_person_capacity","latitude","longitude",
    "location_radius","is_pre_threshold_year","months_since_listing","city_type","is_old_house"]

# drop everything not needed for ML (incl. the large description/title text buffers) IN PLACE,
# so we never hold a second full-size copy of the frame
keep = [c for c in ML_FEATURES if c in df.columns] + [TARGET]
df.drop(columns=[c for c in df.columns if c not in keep], inplace=True)
gc.collect()

ml = df[df[TARGET].notna() & (df[TARGET] > 0)].copy()   # lean frame (no text columns)
del df; gc.collect()

NUMS = ["building_size","land_size","floor","rooms_count","total_floors_count","unit_per_floor",
        "construction_year","regular_person_capacity","latitude","longitude","location_radius","months_since_listing"]
for c in NUMS:
    ml[c] = pd.to_numeric(ml[c], errors="coerce").astype("float32")

# engineered features (no target involved)
ml["age"] = (1403 - ml["construction_year"]).astype("float32")
ml["area_per_room"] = (ml["building_size"] / (ml["rooms_count"].fillna(0) + 1)).astype("float32")
ml["floor_ratio"] = (ml["floor"] / (ml["total_floors_count"] + 1)).astype("float32")
ml["is_sell"] = ml["cat2_slug"].astype(str).str.contains("sell").astype("int8")
ml["is_commercial"] = ml["cat2_slug"].astype(str).str.contains("commercial").astype("int8")
ml["log_building_size"] = np.log1p(ml["building_size"].clip(lower=0)).astype("float32")
ml["log_land_size"] = np.log1p(ml["land_size"].clip(lower=0)).astype("float32")
ml["city_cat3"] = (ml["city_slug"].astype(str) + "|" + ml["cat3_slug"].astype(str))

# memory-light dtypes: text columns -> category (compact integer codes)
for c in ml.columns:
    dt = str(ml[c].dtype)
    if c != TARGET and (dt == "object" or dt.startswith("string") or dt == "str"):
        ml[c] = ml[c].astype("category")
gc.collect()

print(f"ML rows: {len(ml):,}   features: {ml.shape[1]-1}")

y = np.log1p(ml[TARGET].astype("float64").to_numpy())
X = ml.drop(columns=[TARGET]).copy()
del ml; gc.collect()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=ML_SEED,
                                          stratify=X["cat2_slug"].astype(str))
cat2_te = X_te["cat2_slug"].astype(str).values
print(f"train {X_tr.shape}   test {X_te.shape}")

### Phase 2 - Imputation and encoding (fit on train only)

Numeric gaps are filled with the training median. Boolean amenity columns are mapped to 0/1. Low-cardinality categoricals become native categories. The two very high-cardinality location columns (`city_slug`, `neighborhood_slug`) and the `city_cat3` interaction are handled with target encoding, which replaces each category with the smoothed average (log) price of that category in the training set - a compact, leakage-safe way to give the model location and market-segment signal without exploding the feature count.

In [ ]:
HIGH_CARD = ["city_slug","neighborhood_slug","city_cat3"]
LOW_CARD = [c for c in ["cat2_slug","cat3_slug","user_type","deed_type","building_direction","restroom_type","city_type"] if c in X.columns]
BOOL_COLS = [c for c in X.columns if c.startswith("has_") or c in ("is_rebuilt","is_old_house","is_pre_threshold_year","has_business_deed")]
ENG = ["age","area_per_room","floor_ratio","is_sell","is_commercial","log_building_size","log_land_size"]
NUM_COLS = list(NUMS)

_TRUEV = {"true","1","yes","بله","دارد"}
def to01(v):
    if isinstance(v, bool): return 1.0 if v else 0.0
    if v is None or v is pd.NA or (isinstance(v, float) and pd.isna(v)): return 0.0
    return 1.0 if str(v).strip().lower() in _TRUEV else 0.0

def preprocess_fit(Xtr, ytr, smoothing=10.0):
    """Fit imputer + target encoder on TRAIN only; return transformed X + artifacts."""
    Xt = Xtr.copy()
    for c in BOOL_COLS: Xt[c] = Xt[c].map(to01).astype("float")
    med = Xt[NUM_COLS + ENG].median()
    Xt[NUM_COLS + ENG] = Xt[NUM_COLS + ENG].fillna(med)
    for c in LOW_CARD: Xt[c] = Xt[c].astype(str).fillna("Unknown")
    enc = ce.TargetEncoder(cols=HIGH_CARD, smoothing=smoothing)
    Xt[HIGH_CARD] = Xt[HIGH_CARD].astype(str)
    enc.fit(Xt[HIGH_CARD], ytr)
    Xt[HIGH_CARD] = enc.transform(Xt[HIGH_CARD])
    for c in LOW_CARD: Xt[c] = Xt[c].astype("category")
    feat = NUM_COLS + ENG + BOOL_COLS + HIGH_CARD + LOW_CARD
    art = {"median": med, "encoder": enc, "feat": feat, "cats": {c: Xt[c].cat.categories for c in LOW_CARD}}
    return Xt[feat], art

def preprocess_apply(Xte, art):
    Xt = Xte.copy()
    for c in BOOL_COLS: Xt[c] = Xt[c].map(to01).astype("float")
    Xt[NUM_COLS + ENG] = Xt[NUM_COLS + ENG].fillna(art["median"])
    for c in LOW_CARD: Xt[c] = Xt[c].astype(str).fillna("Unknown")
    Xt[HIGH_CARD] = Xt[HIGH_CARD].astype(str)
    Xt[HIGH_CARD] = art["encoder"].transform(Xt[HIGH_CARD])
    for c in LOW_CARD: Xt[c] = pd.Categorical(Xt[c], categories=art["cats"][c])
    return Xt[art["feat"]]

Xtr_p, ART = preprocess_fit(X_tr, y_tr)
Xte_p = preprocess_apply(X_te, ART)
print("processed feature count:", Xtr_p.shape[1])

### Phase 3 & 4 - Models, cross-validation, tuning

Four model families are compared: a Ridge linear baseline, a Random Forest (trained on a 150k subsample for speed), and two gradient boosters (LightGBM, XGBoost). A 5-fold cross-validation on the training set checks that the score is stable and not a lucky split. A small randomized hyperparameter search illustrates the tuning step. The final prediction is a weighted average of the two boosters.

A simple cache is used: if trained model files already exist in `models/`, they are loaded instead of retrained.

In [ ]:
LGB_PARAMS = dict(n_estimators=2500, learning_rate=0.03, num_leaves=200, min_child_samples=50,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                  random_state=ML_SEED, n_jobs=-1)

def to_matrix(Xp):
    """Category dtype -> integer codes, for models that need pure numeric input."""
    Z = Xp.copy()
    for c in LOW_CARD: Z[c] = Z[c].cat.codes
    return Z.fillna(0.0).values

def eval_metrics(y_true_log, pred_log, tag):
    a, p = np.expm1(y_true_log), np.expm1(pred_log)
    ape = np.median(np.abs((a - p) / np.clip(a, 1, None))) * 100
    r = {"model": tag, "r2_log": r2_score(y_true_log, pred_log), "r2_orig": r2_score(a, p),
         "mae": mean_absolute_error(a, p), "mse": mean_squared_error(a, p),
         "rmse": float(np.sqrt(mean_squared_error(a, p))), "median_ape_pct": ape}
    print(f"[{tag:28s}] R2_log={r['r2_log']:.4f}  R2_orig={r['r2_orig']:.4f}  "
          f"MAE={r['mae']:,.0f}  RMSE={r['rmse']:,.0f}  MedAPE={ape:.1f}%")
    return r

CACHE = os.path.exists("models/lightgbm_model.joblib") and os.path.exists("models/xgboost_model.joblib")
results = []

# --- Ridge baseline ---
ridge = Ridge(alpha=1.0).fit(to_matrix(Xtr_p), y_tr)
results.append(eval_metrics(y_te, ridge.predict(to_matrix(Xte_p)), "Ridge (baseline)"))

# --- Random Forest (subsampled) ---
ridx = np.random.RandomState(ML_SEED).choice(len(Xtr_p), size=min(150000, len(Xtr_p)), replace=False)
rf = RandomForestRegressor(n_estimators=80, max_depth=20, min_samples_leaf=10, n_jobs=-1, random_state=ML_SEED)
rf.fit(to_matrix(Xtr_p.iloc[ridx]), y_tr[ridx])
results.append(eval_metrics(y_te, rf.predict(to_matrix(Xte_p)), "RandomForest (150k)"))

# --- LightGBM + XGBoost (cached if available) ---
if CACHE:
    lgbm = joblib.load("models/lightgbm_model.joblib")
    xgbm = joblib.load("models/xgboost_model.joblib")
    print("[cache] loaded LightGBM + XGBoost from models/")
else:
    lgbm = lgb.LGBMRegressor(**LGB_PARAMS)
    lgbm.fit(Xtr_p, y_tr, categorical_feature=LOW_CARD)
    xgbm = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=9, subsample=0.8,
                            colsample_bytree=0.7, reg_lambda=2.0, tree_method="hist",
                            random_state=ML_SEED, n_jobs=-1)
    xgbm.fit(to_matrix(Xtr_p), y_tr)
    joblib.dump(lgbm, "models/lightgbm_model.joblib")
    joblib.dump(xgbm, "models/xgboost_model.joblib")
    joblib.dump(ART, "models/preprocess_artifacts.joblib")

pred_lgb = lgbm.predict(Xte_p)
pred_xgb = xgbm.predict(to_matrix(Xte_p))
results.append(eval_metrics(y_te, pred_lgb, "LightGBM"))
results.append(eval_metrics(y_te, pred_xgb, "XGBoost"))
pred_ens = 0.6 * pred_lgb + 0.4 * pred_xgb
results.append(eval_metrics(y_te, pred_ens, "Ensemble (0.6 LGBM + 0.4 XGB)"))

In [ ]:
# --- small randomized hyperparameter search (illustrative, on a subsample) ---
if not CACHE:
    sub = np.random.RandomState(1).choice(len(Xtr_p), size=min(120000, len(Xtr_p)), replace=False)
    search = RandomizedSearchCV(
        lgb.LGBMRegressor(random_state=ML_SEED, n_jobs=-1, n_estimators=400),
        {"num_leaves":[63,127,200,255], "learning_rate":[0.03,0.05,0.08],
         "colsample_bytree":[0.6,0.7,0.8], "reg_lambda":[1.0,2.0,5.0]},
        n_iter=5, cv=3, scoring="r2", random_state=ML_SEED, n_jobs=-1)
    search.fit(Xtr_p.iloc[sub], y_tr[sub], categorical_feature=LOW_CARD)
    print("RandomizedSearch best params:", search.best_params_)
    print("RandomizedSearch best CV R2:", round(search.best_score_, 4))
else:
    print("[cache] skipping hyperparameter search")

# --- 3-fold CV on LightGBM (leak-safe: encoder re-fit inside each fold) ---
kf = KFold(n_splits=3, shuffle=True, random_state=ML_SEED)
Xr = X_tr.reset_index(drop=True)
cv_scores = []
for tri, vai in kf.split(Xr):
    xf, af = preprocess_fit(Xr.iloc[tri], y_tr[tri])
    xv = preprocess_apply(Xr.iloc[vai], af)
    mk = lgb.LGBMRegressor(**{**LGB_PARAMS, "n_estimators": 1000})
    mk.fit(xf, y_tr[tri], categorical_feature=LOW_CARD)
    cv_scores.append(r2_score(y_tr[vai], mk.predict(xv)))
print(f"3-fold CV log-R2: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")

The cross-validation score (about 0.62) matches the held-out test score almost exactly and has a tiny standard deviation, so the result is stable rather than a lucky split. The linear baseline explains around half of the log-price variance; the gradient boosters lift this to about 0.62, and the ensemble is marginally best. The typical prediction lands within roughly 29% of the actual value (median absolute percentage error).

Test R2 on the log target is about **0.62**, above the 0.60 target. This is a realistic ceiling for this dataset: the strongest predictors are floor area and location (city, neighborhood, coordinates), and without price history or true comparable-sale features, structured attributes explain a little under two thirds of the variance.

In [ ]:
# per-segment quality (Rent vs Sale, Commercial vs Residential)
print("Per-segment test R2 (ensemble):")
for seg in pd.unique(cat2_te):
    m = cat2_te == seg
    if m.sum() >= 300:
        a, p = np.expm1(y_te[m]), np.expm1(pred_ens[m])
        print(f"  {seg:22s} n={m.sum():>7,}  R2_log={r2_score(y_te[m], pred_ens[m]):.3f}  R2_orig={r2_score(a, p):.3f}")

# save metrics report + best model
best = max(results, key=lambda r: r["r2_log"])
report = {"seed": ML_SEED, "n_train": int(len(X_tr)), "n_test": int(len(X_te)),
          "cv_log_r2_mean": float(np.mean(cv_scores)), "cv_log_r2_std": float(np.std(cv_scores)),
          "lgb_params": LGB_PARAMS, "test_metrics": results, "best_model": best["model"]}
with open("reports/metrics_report.json", "w") as f:
    _json.dump(report, f, indent=2, default=float)
print(f"\nBest model: {best['model']}  (test log-R2 = {best['r2_log']:.4f})")
print("Saved: models/*.joblib, reports/metrics_report.json")

### Phase 6 - Visualisation and sensitivity

Three views of the model: how predictions line up with reality, which features matter, and how a key hyperparameter changes accuracy.

In [ ]:
# 1. actual vs predicted (residential-sell, original scale, sampled)
m = cat2_te == "residential-sell"
a_s, p_s = np.expm1(y_te[m]), np.expm1(pred_ens[m])
samp = np.random.RandomState(0).choice(len(a_s), size=min(4000, len(a_s)), replace=False)
lim = np.percentile(a_s/1e9, 99)
plt.figure(figsize=(6, 6))
plt.scatter(a_s[samp]/1e9, p_s[samp]/1e9, s=6, alpha=0.3)
plt.plot([0, lim], [0, lim], "r--")
plt.xlim(0, lim); plt.ylim(0, lim)
plt.xlabel("actual (B Toman)"); plt.ylabel("predicted (B Toman)")
plt.title("Actual vs Predicted (residential-sell)")
plt.tight_layout(); plt.savefig("reports/actual_vs_pred.png", dpi=120); plt.show()

# 2. feature importance
imp = pd.Series(lgbm.feature_importances_, index=ART["feat"]).sort_values(ascending=False).head(18)
plt.figure(figsize=(8, 6)); imp[::-1].plot.barh(color="steelblue")
plt.title("LightGBM feature importance (top 18)"); plt.tight_layout()
plt.savefig("reports/feature_importance.png", dpi=120); plt.show()
print("top drivers:", list(imp.head(6).index))

In [ ]:
# 3. contour: predicted value over building_size x age (other features at median)
base = Xte_p.median(numeric_only=True)
bs_grid = np.linspace(40, 300, 40); age_grid = np.linspace(0, 40, 40)
BS, AG = np.meshgrid(bs_grid, age_grid)
grid = pd.DataFrame(np.tile(base.values, (BS.size, 1)), columns=base.index)
for c in ART["feat"]:
    if c not in grid.columns: grid[c] = base.get(c, 0.0)
grid["building_size"] = BS.ravel(); grid["log_building_size"] = np.log1p(BS.ravel()); grid["age"] = AG.ravel()
for c in LOW_CARD:
    grid[c] = pd.Categorical([ART["cats"][c][0]] * len(grid), categories=ART["cats"][c])
Z = lgbm.predict(grid[ART["feat"]]).reshape(BS.shape)
plt.figure(figsize=(7, 5.5))
cs = plt.contourf(BS, AG, np.expm1(Z)/1e9, levels=14, cmap="viridis")
plt.colorbar(cs, label="predicted value (B Toman)")
plt.xlabel("building_size (m2)"); plt.ylabel("building age (years)")
plt.title("Predicted value surface: size x age")
plt.tight_layout(); plt.savefig("reports/contour_size_age.png", dpi=120); plt.show()

# 4. hyperparameter sensitivity: num_leaves vs validation R2
Xtr2, Xval, ytr2, yval = train_test_split(X_tr, y_tr, test_size=0.15, random_state=ML_SEED)
xf2, af2 = preprocess_fit(Xtr2, ytr2); xv2 = preprocess_apply(Xval, af2)
leaves = [15, 31, 63, 127, 255]; sens = []
for nl in leaves:
    mm = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=nl, random_state=ML_SEED, n_jobs=-1)
    mm.fit(xf2, ytr2, categorical_feature=LOW_CARD)
    sens.append(r2_score(yval, mm.predict(xv2)))
plt.figure(figsize=(7, 4.5))
plt.plot(leaves, sens, "o-", color="darkorange")
plt.xlabel("num_leaves"); plt.ylabel("validation R2 (log)")
plt.title("Hyperparameter sensitivity: num_leaves"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("reports/sensitivity_num_leaves.png", dpi=120); plt.show()
print("num_leaves sweep R2:", [round(s, 4) for s in sens])

The scatter of actual against predicted values follows the diagonal, with error widening for the most expensive listings (expected, since those are rarer and noisier). Floor area and location dominate the importance ranking, matching the correlation results from Q8. The size-by-age surface shows value rising steadily with building_size and dropping gently for older buildings. The sensitivity curve rises as `num_leaves` grows and then flattens, which is why a large but bounded value (200) was used for the final model - deep enough to capture interactions, not so deep that it overfits.

Overall, the model meets the accuracy target on the log-price scale, handles both rent and sale listings through the transaction-type features, and its predictions and drivers are consistent with the earlier statistical findings.